# BGE-M3 Fine-Tune v2 — 10.6k pairs (4x previous dataset)

Fine-tunes `BAAI/bge-m3` on Roman-Urdu/English anchor→positive product-search
pairs, using LoRA (rank-32) on the attention layers, then merges the LoRA
weights into the base model so the saved checkpoint is a plain
`sentence-transformers` model — no PEFT dependency needed to load and use it
later (this is the model later uploaded to
[huggingface.co/muskannnnn/Prototype](https://huggingface.co/muskannnnn/Prototype)).

**Dataset:** 10,640 pairs, 6,245 unique anchors, 2,128 products (previous run:
2,500 pairs → MRR@20 62.2%; this run targets a larger lift from 4x the data).

**Why LoRA instead of a full fine-tune:** cheaper to train, less prone to
catastrophic forgetting of the base model's general multilingual
understanding, and produces a small adapter that's easy to re-run with
different hyperparameters. It's merged back into the base weights at the end
purely for deployment convenience — the saved model doesn't carry a PEFT
dependency.

This notebook bakes in three fixes discovered the hard way in the previous
run, each explained inline where it matters:
- `torchao` must be uninstalled — it version-clashes with `peft`
- `LoraModel` is used directly, not `get_peft_model` — the latter silently
  does nothing on a `sentence-transformers` model, so training would run
  without error but touch zero LoRA weights
- The LoRA-wrapped model is merged **manually** after training, because
  `sentence-transformers`' `.fit()` discards the PEFT wrapper object (though
  the trained weights themselves do survive inside it — see the checks in
  the training cell below)
- The tokenizer and vocab file are saved explicitly, since `model.save()`
  alone can leave `special_tokens_map.json` missing

## 1. Install dependencies

`torchao` is uninstalled immediately after install — if a newer version
somehow finds its way back in mid-session, the environment check in the next
cell will catch it and tell you to restart the kernel rather than let LoRA
injection fail silently later.

In [ ]:
!pip install -q sentence-transformers faiss-cpu numpy peft
!pip uninstall -y -q torchao 2>/dev/null || true

## 2. Paths + environment check

Fails fast (before any GPU time is spent) if a required dataset file is
missing, or if `torchao` snuck back in at a version that will break `peft`.

In [ ]:
import os, importlib.util

# ---- EDIT THESE PATHS ----
GROUPED_TRAIN_JSON = "/kaggle/input/datasets/ameerhvmza/completedataset/grouped_training_data_v2.json"
HELD_OUT_EVAL_CSV  = "/kaggle/input/datasets/ameerhvmza/evaluationcomplete/held_out_eval_v2.csv"
FULL_CATALOG_CSV   = "/kaggle/input/datasets/ameerhvmza/allproducts/npk_prods_dmp.csv"

for name, path in [("TRAIN", GROUPED_TRAIN_JSON), ("EVAL", HELD_OUT_EVAL_CSV), ("CATALOG", FULL_CATALOG_CSV)]:
    ok = os.path.exists(path)
    print(f"  {name}: {'OK' if ok else 'MISSING'} -> {path}")
    if not ok:
        raise FileNotFoundError(f"{name} not found at {path}")

if importlib.util.find_spec("torchao") is not None:
    import torchao
    from packaging import version
    if version.parse(torchao.__version__) < version.parse("0.16.0"):
        raise ImportError(
            f"torchao {torchao.__version__} still installed. Restart kernel, re-run from top.")
    print(f"torchao {torchao.__version__} - OK")
else:
    print("torchao removed - clean.")

## 3. Evaluation function

Measures retrieval quality the same way the app will actually use the model:
embed the **full** catalogue (`npk_prods_dmp.csv`, ~155k products, not just
the 2,128 trained-on products) into a FAISS flat index, then for each
held-out `(query, target_name)` pair, encode the query and check what rank
the correct product comes back at.

- **MRR@20** (Mean Reciprocal Rank) — averages `1/rank` across all queries
  (0 if the target isn't in the top 20 at all). Rewards getting the right
  answer *near the top*, not just somewhere in the list.
- **Hit@20** — the simpler "was the correct product anywhere in the top 20"
  rate.

Evaluating against the full catalogue (not just the training products) is
what makes this number meaningful — it's testing whether the model can find
the right product among ~155k real distractors, which is the actual task at
inference time.

In [ ]:
import pandas as pd
import numpy as np
import faiss

def evaluate(model, full_catalog_csv, held_out_eval_csv, top_k=20):
    full = pd.read_csv(full_catalog_csv, low_memory=False)
    full["text_to_embed"] = (
        full["name"].fillna("") + " " + full["brand"].fillna("") + " "
        + full["category_hierarchy"].fillna("")
    )
    product_names = full["name"].tolist()
    print(f"Embedding {len(product_names):,} catalogue products...")
    product_embeddings = model.encode(
        full["text_to_embed"].tolist(), convert_to_numpy=True,
        normalize_embeddings=True, show_progress_bar=True, batch_size=256)
    index = faiss.IndexFlatIP(product_embeddings.shape[1])
    index.add(np.asarray(product_embeddings, dtype="float32"))

    eval_df = pd.read_csv(held_out_eval_csv)
    results = []
    for _, row in eval_df.iterrows():
        q_emb = model.encode([row["query"]], convert_to_numpy=True, normalize_embeddings=True)
        _, indices = index.search(np.asarray(q_emb, dtype="float32"), top_k)
        rank = 0
        for i, idx in enumerate(indices[0]):
            if idx < len(product_names) and product_names[idx] == row["target_name"]:
                rank = i + 1
                break
        results.append({"query": row["query"], "target": row["target_name"], "rank": rank})

    df = pd.DataFrame(results)
    df["rr"] = df["rank"].apply(lambda r: 1/r if r > 0 else 0)
    mrr = df["rr"].mean() * 100
    hits = (df["rank"] > 0).mean() * 100
    print(f"  MRR@{top_k}: {mrr:.1f}%  |  Hit@{top_k}: {hits:.1f}%")
    misses = df[df["rank"] == 0]
    if len(misses):
        print(f"  {len(misses)}/{len(df)} queries missed target in top {top_k}")
    df.to_csv("heldout_results.csv", index=False)
    return df

## 4. Training dataset

`grouped_training_data_v2.json` maps each anchor (a query) to a list of one
or more positive products. `GroupedAnchorDataset` treats each **anchor** as
one training example per epoch (not each pair) — if an anchor has multiple
valid positives, `_cursor` cycles through them round-robin across epochs
rather than randomly, so over enough epochs every positive gets seen roughly
equally without needing to expand the dataset into a much larger flat pair
list.

`set_epoch()` reshuffles the anchor order using a seed derived from the
epoch number, so each epoch sees a different batch order (this replaces
`DataLoader(shuffle=True)`, which the training loop below deliberately
doesn't use — see the note in cell 6).

In [ ]:
import json, random
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import InputExample

class GroupedAnchorDataset(Dataset):
    def __init__(self, path):
        with open(path, encoding="utf-8") as f:
            self.anchor_to_positives = json.load(f)
        self.anchors = list(self.anchor_to_positives.keys())
        self._cursor = {a: 0 for a in self.anchors}
        n_pairs = sum(len(v) for v in self.anchor_to_positives.values())
        n_multi = sum(1 for v in self.anchor_to_positives.values() if len(v) > 1)
        print(f"Loaded {n_pairs:,} pairs -> {len(self.anchors):,} unique anchors "
              f"({n_multi:,} with >1 positive)")

    def set_epoch(self, epoch):
        random.Random(epoch).shuffle(self.anchors)

    def __len__(self):
        return len(self.anchors)

    def __getitem__(self, idx):
        anchor = self.anchors[idx]
        positives = self.anchor_to_positives[anchor]
        i = self._cursor[anchor] % len(positives)
        self._cursor[anchor] += 1
        return InputExample(texts=[anchor, positives[i]])

def make_dataloader(dataset, batch_size, epoch):
    dataset.set_epoch(epoch)
    return DataLoader(dataset, batch_size=batch_size, shuffle=False)

## 5. Save-verification + LoRA merge helpers

`verify_saved_model()` is a deliberately paranoid check — it runs *after*
`model.save()` and confirms every file `SentenceTransformer(...)` needs to
reload cleanly is actually on disk, including the Pooling module's
`config.json`. Better to fail loudly here than discover a broken checkpoint
after uploading a 2GB zip.

`merge_lora_weights()` walks every module looking for LoRA layers (anything
with a `base_layer` + a callable `.merge()`, which is how `peft`'s LoRA
layers are structured) and merges each one's learned delta into the base
weight, replacing the LoRA-wrapped module with the plain merged one in the
parent. It raises if it merged zero layers — silently training nothing and
saving the untouched base model would otherwise look exactly like success.

In [ ]:
import os, shutil

REQUIRED_FILES = [
    "config.json", "config_sentence_transformers.json", "model.safetensors",
    "modules.json", "sentence_bert_config.json", "tokenizer.json",
    "tokenizer_config.json", os.path.join("1_Pooling", "config.json"),
]

def verify_saved_model(path):
    missing = [f for f in REQUIRED_FILES if not os.path.exists(os.path.join(path, f))]
    if missing:
        raise RuntimeError(f"Save incomplete for '{path}'. Missing: {missing}")
    print(f"Verified: all {len(REQUIRED_FILES)} required files present in '{path}'.")

def merge_lora_weights(module):
    merged = 0
    for name, m in list(module.named_modules()):
        if hasattr(m, "base_layer") and hasattr(m, "merge") and callable(m.merge):
            m.merge()
            *parents, leaf = name.split(".")
            parent = module
            for p in parents:
                parent = getattr(parent, p)
            setattr(parent, leaf, m.base_layer)
            merged += 1
    if merged == 0:
        raise RuntimeError("merge_lora_weights found 0 LoRA layers to merge.")
    print(f"Merged {merged} LoRA layers into base weights.")
    return module

## 6. Training function

Step by step:

1. **Load base `BAAI/bge-m3`** fresh each run (not the previous run's
   checkpoint) so different hyperparameter configs are independent
   experiments, not compounding fine-tunes.
2. **Discover attention module names** dynamically rather than hardcoding
   them — different model internals name their query/key/value projections
   differently (`query`/`key`/`value` vs `q_proj`/`k_proj`/`v_proj`), so this
   prints what's actually present as a sanity check before LoRA targets them.
3. **Inject LoRA via `LoraModel` directly** — `peft.get_peft_model()` is the
   usual entry point, but on a bare `AutoModel` wrapped inside
   `sentence-transformers`, it was found to silently return a model with no
   trainable LoRA parameters at all (no error, no warning — the fix here
   was purely found by noticing the `lora_params` check below coming back
   empty in an earlier attempt). Calling `LoraModel(...)` directly on the
   inner transformer skips whatever `get_peft_model` was doing wrong.
4. **Verify injection actually worked** before spending any GPU time
   training — checks `lora_` parameters exist and prints the trainable
   parameter percentage (should be a small fraction of the total; that's
   the point of LoRA).
5. **Train one epoch at a time in a loop**, not `epochs=N` in one `.fit()`
   call — this is what lets `make_dataloader()` call `dataset.set_epoch()`
   between epochs to reshuffle anchors, since `.fit()` doesn't expose a
   per-epoch hook for that.
6. **Verify LoRA weights survived `.fit()`** — `sentence-transformers`'
   `.fit()` was found to discard the `LoraModel` wrapper object itself
   (`type(auto).__name__` won't be `LoraModel` afterward), but the actual
   trained tensors remain in the underlying module's `state_dict()` under
   `lora_` keys. This check confirms training wasn't silently a no-op.
7. **Save an adapter backup** (just the LoRA delta weights + their config)
   before merging, in case the merge step ever needs to be redone or
   debugged separately from the full merged checkpoint.
8. **Merge LoRA into the base weights, save, verify, zip.**

In [ ]:
import torch
from sentence_transformers import SentenceTransformer, losses
from peft import LoraConfig, TaskType
from peft.tuners.lora import LoraModel

def train(config, dataset):
    print(f"\n{'='*60}")
    print(f"  Run {config['id']}: epochs={config['epochs']}  bs={config['batch_size']}  "
          f"lr={config['learning_rate']}  lora_r={config.get('lora_r', 8)}")
    print(f"{'='*60}\n")

    model = SentenceTransformer("BAAI/bge-m3")
    base_vocab_file = getattr(model.tokenizer, "vocab_file", None)

    # Discover module names
    attn_names = sorted({
        n.rsplit(".", 1)[-1] for n, _ in model[0].auto_model.named_modules()
        if any(k in n.lower() for k in ["query", "key", "value", "q_proj", "k_proj", "v_proj"])
    })
    print(f"Attention module names: {attn_names}")

    # Wrap with LoRA
    peft_config = LoraConfig(
        task_type=TaskType.FEATURE_EXTRACTION, inference_mode=False,
        r=config.get("lora_r", 8), lora_alpha=32, lora_dropout=0.1,
        target_modules=["query", "key", "value", "q_proj", "k_proj", "v_proj"],
    )
    lora_model = LoraModel(model[0].auto_model, {"default": peft_config}, "default")
    model[0].auto_model = lora_model

    # Verify injection
    lora_params = [n for n, _ in model[0].auto_model.named_parameters() if "lora_" in n]
    if not lora_params:
        raise RuntimeError(f"No lora_ params found. Use these names: {attn_names}")
    trainable = sum(p.numel() for p in model[0].auto_model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model[0].auto_model.parameters())
    print(f"trainable: {trainable:,} / {total:,} ({100*trainable/total:.4f}%)")
    print(f"LoRA tensors: {len(lora_params)}\n")

    # Train
    train_loss = losses.MultipleNegativesRankingLoss(model)
    for epoch in range(config["epochs"]):
        dl = make_dataloader(dataset, config["batch_size"], epoch)
        model.fit(
            train_objectives=[(dl, train_loss)], epochs=1,
            warmup_steps=10 if epoch == 0 else 0,
            optimizer_params={"lr": config["learning_rate"]}, use_amp=True)
    print("\nTraining complete.\n")

    # Verify LoRA survived .fit()
    auto = model[0].auto_model
    lora_after = {k: v for k, v in auto.state_dict().items() if "lora_" in k}
    print(f"After .fit(): {type(auto).__name__}, LoRA tensors: {len(lora_after)}")
    if not lora_after:
        raise RuntimeError("LoRA weights gone after .fit()")

    save_path = f"model_run_{config['id']}"

    # Adapter backup
    adapter_path = f"{save_path}_lora_adapter"
    os.makedirs(adapter_path, exist_ok=True)
    torch.save(lora_after, os.path.join(adapter_path, "adapter_model.bin"))
    peft_config.save_pretrained(adapter_path)
    print(f"Adapter backup: {len(lora_after)} tensors -> {adapter_path}/")

    # Merge + save
    merge_lora_weights(auto)
    model[0].auto_model = auto
    model.save(save_path)
    model.tokenizer.save_pretrained(save_path)

    if base_vocab_file and os.path.exists(base_vocab_file):
        dest = os.path.join(save_path, os.path.basename(base_vocab_file))
        if not os.path.exists(dest):
            shutil.copy(base_vocab_file, dest)

    verify_saved_model(save_path)
    shutil.make_archive(save_path, "zip", save_path)
    print(f"\nDone: {save_path}.zip")
    return model

## 7. Run it

`BEST_CONFIG` (`id=6`, `epochs=10`, `batch_size=8`, `lr=5e-5`, `lora_r=32`)
is the hyperparameter combination that produced the best MRR@20 across the
runs tried — a higher LoRA rank (32 vs the LoRA default of 8) gives the
adapter more capacity to fit the larger 10.6k-pair dataset than the previous
2,500-pair run needed.

Running this trains the model, saves it as `model_run_6/` (+ a zip of the
same), and immediately evaluates it against the held-out set — the printed
MRR@20/Hit@20 is the number to compare against the previous run's 62.2%/66.2%
baseline.

`model_run_6/` (or its zip) is what later gets uploaded to
`huggingface.co/muskannnnn/Prototype`, and what the embedding-generation
notebook loads to encode the catalogue.

In [ ]:
dataset = GroupedAnchorDataset(GROUPED_TRAIN_JSON)

BEST_CONFIG = {"id": 6, "epochs": 10, "batch_size": 8, "learning_rate": 5e-5, "lora_r": 32}

model = train(BEST_CONFIG, dataset)

evaluate(model, FULL_CATALOG_CSV, HELD_OUT_EVAL_CSV)